In [1]:
# IMPORT THE DATASETS

In [ ]:
import numpy as np
import subprocess
import pandas as pd
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import sys
from pathlib import Path
import os

root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))
print(f"Setting root of the project to: {root}")

def alleleFrequency(base, allele_count):
    # Allele count format: [A : T : C : G : N : del]
    base_dict = {
        'A': 0,
        'T': 1,
        'C': 2,
        'G': 3,
        'N': 4,   # unused
        'del': 5  # unsued, both defined to be explicit
    }
    base_index = base_dict.get(base)
    vector = [int(num) for num in allele_count.split(':')]

    # Sum all values
    total = sum(vector)

    # Error-handle for division-by-zero
    if total == 0:
        return 0.0

    # Find difference between base & total
    base_value = vector[base_index]
    mutations = total - base_value
    freq = mutations / total
    return freq, mutations, total
    
def readAlleleFrequencyData(file_path="../data/chromosome_2L.sync"):
    columns = [
        'chromosome',
        'position',
        'base',
        'Dsim_Fl_Base_1',
        'Dsim_Fl_Hot_F10_1',
        'Dsim_Fl_Hot_F20_1', 
        'Dsim_Fl_Hot_F30_1',
        'Dsim_Fl_Hot_F40_1', 
        'Dsim_Fl_Hot_F50_1', 
        'Dsim_Fl_Hot_F60_1'
    ]

    df = pd.read_csv(
        filepath_or_buffer=file_path,
        sep='\t', # tab-separated values
        names=columns,
        usecols=columns,
        header=None # no headers provided
    )

    rename_dict = {
        'Dsim_Fl_Base_1':    'gen_0',
        'Dsim_Fl_Hot_F10_1': 'gen_10',
        'Dsim_Fl_Hot_F20_1': 'gen_20',
        'Dsim_Fl_Hot_F30_1': 'gen_30',
        'Dsim_Fl_Hot_F40_1': 'gen_40',
        'Dsim_Fl_Hot_F50_1': 'gen_50',
        'Dsim_Fl_Hot_F60_1': 'gen_60'
    }

    # Rename the columns (inplace=True modifies the original DataFrame)
    df.drop(columns=['chromosome'], inplace=True)
    df.rename(columns=rename_dict, inplace=True)

    for column in ['gen_0','gen_10','gen_20','gen_30','gen_40','gen_50','gen_60']:
        df[[column, f'mut_{column}', f'total_{column}']] = [
            alleleFrequency(b, c) 
            for b, c in zip(df['base'], df[column])
        ]
        
    for generation in [10,20,30,40,50,60]:
        prev_generation = generation - 10
        df[f'delta_{generation}'] = df[f'gen_{generation}'] - df[f'gen_{prev_generation}']

    return df

# def singleHypergeometricSample(num_mutations, actual_total=401, seed=42):
#     rng = np.random.default_rng(seed=seed)
#     num_non_mutations = 2000 - num_mutations
#     return rng.hypergeometric(size=len(num_mutations), ngood=num_mutations, nbad=num_non_mutations, nsample=actual_total)

# def readSamples():
#     dir_path = "../slim/outputs/"
#     file_paths = []
#     for filename in os.listdir(dir_path):
#         full_path = os.path.join(dir_path, filename)
#         if os.path.isfile(full_path):
#             file_paths.append(full_path)

#     df_list = []
#     print('reading files')
#     for file_path in file_paths:
#         tmp = pd.read_csv(file_path)
#         tmp.drop(columns=['Unnamed: 0'], inplace=True)
#         df_list.append(tmp)

#     print('concatenating dataframes')
#     df = pd.concat(df_list, ignore_index=True)

#     print('hypergeometric sampling')
#     for t in [10, 20, 30, 40, 50, 60]:
#         df.loc[:, f'sim_{t}'] = singleHypergeometricSample(num_mutations=df[f'sim_{t}'])
    
#     return df

# def readSamplesPolars():
#     # Polars can scan and read multiple files simultaneously
#     file_pattern = "../slim/outputs/*.csv" 
    
#     # read_csv accepts wildcards or lists of paths natively
#     df = pl.read_csv(file_pattern)
    
#     # Drop the column if it exists
#     if "Unnamed: 0" in df.columns:
#         df = df.drop("Unnamed: 0")

#     for t in [10, 20, 30, 40, 50, 60]:
#         df.loc[:, f'sim_{t}'] = singleHypergeometricSample(num_mutations=df[f'sim_{t}'])
        
#     return df # Can convert back via df.to_pandas() if needed!

def singleHypergeometricSample(num_mutations, rng, actual_total=401):
    num_mutations = np.asarray(num_mutations, dtype=np.int64)
    num_non_mutations = 2000 - num_mutations

    return rng.hypergeometric(
        ngood=num_mutations,
        nbad=num_non_mutations,
        nsample=actual_total,
        size=len(num_mutations),
    )
def readSamplesPolars():
    df = pl.read_csv("../slim/outputs/*.csv").drop("Unnamed: 0", strict=False)

    rng = np.random.default_rng(42)

    df = df.with_columns([
        pl.Series(
            f"sim_{t}",
            singleHypergeometricSample(df[f"sim_{t}"].to_numpy(), rng)
        )
        for t in [10, 20, 30, 40, 50, 60]
    ])


sim = readSamplesPolars()
sim
# print(sim.head())

# df = readAlleleFrequencyData()
# print(df.head())

Setting root of the project to: /home/kv/education/predicting_DFE


In [ ]:
# PUT SIMULATIONS IN FINAL FORMAT
sns.histplot(data=sim, x='sim_10')

In [1]:
DL = sim[['position', 'alpha', 'beta', 'sim_10', sim_20', 'sim_30', 'sim_40', 'sim_50', 'sim_60']].copy()
DL = DL.sort_values(['alpha', 'beta', 'position'])
DL['norm_position'] = DL.groupby(['alpha', 'beta']).cumcount() + 1

id_vars = ['norm_position', 'alpha', 'beta']
value_vars = ['sim_10', 'sim_20', 'sim_30', 'sim_40', 'sim_50', 'sim_60']
DM = DL.melt(id_vars=id_vars, value_vars=value_vars, var_name='time', value_name='diff')

# Extract time as clean numbers (1 to 6)
DM['time'] = DM['time'].str.replace("sim_", "").str.replace("0", "").astype(int)

# 1. Pivot using numeric variables. Pandas automatically sorts integers numerically!
# Rows: alpha & beta, Columns: norm_position first (1-10000), then time (1-6)
DW = DM.pivot(index=['alpha', 'beta'], columns=['norm_position', 'time'], values='diff')

# 2. Because it's sorted perfectly, we can generate the strings safely now
# This loops through the correctly ordered numeric columns and builds the string names
DW.columns = [f"p{p}_t{t}" for p, t in DW.columns]

# 3. Bring back alpha and beta as regular columns
DW = DW.reset_index()

Processing ../slim/outputs/alpha=47.15_beta=47.15.csv...


TypeError: Col.__call__() missing 1 required positional argument: 'name'

In [ ]:
# PUT ACTUAL DATA IN FINAL FORMAT

# df
DW

In [ ]:
# Pull in real values to test on
dfL = df[['position',
          'mut_gen_10', 'mut_gen_20', 'mut_gen_30',
          'mut_gen_40', 'mut_gen_50', 'mut_gen_60']].copy()

# Take 10,000 random positions
rng = np.random.default_rng(53)
randomPositions = rng.choice(
    dfL['position'].unique(),
    size=10_000,
    replace=False
)

# Keep only those positions and sort them
dfL = (
    dfL[dfL['position'].isin(randomPositions)]
    .sort_values('position')
    .copy()
)

# Create normalized position index
dfL['norm_position'] = np.arange(1, len(dfL) + 1)

# Long format
dfM = dfL.melt(
    id_vars=['norm_position'],
    value_vars=[
        'mut_gen_10', 'mut_gen_20', 'mut_gen_30',
        'mut_gen_40', 'mut_gen_50', 'mut_gen_60'
    ],
    var_name='time',
    value_name='freq'
)

# Extract numeric time
dfM['time'] = (
    dfM['time']
    .str.replace("mut_gen_", "", regex=False)
    .str.replace("0", "", regex=False)
    .astype(int)
)

dfM['dummy'] = 0
dfW = dfM.pivot(
    index='dummy',
    columns=['norm_position', 'time'],
    values='freq'
)

dfW.columns = [f"p{p}_t{t}" for p, t in dfW.columns]
dfW = dfW.reset_index(drop=True)

dfW

In [ ]:
# RUN MODEL

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

responses = ['alpha', 'beta']
Y = DW[responses]
X = DW.drop(columns=responses)

# X: IMAGE ARCHITECTURE
#
# In image recognition, there is a massive vector (or "tensor"), each cell with a value.
# Importantly, you simply feed in the entire vector.
#
# We want:
#     p1 p2 p3 ... p10_000
# t1 | 0  0  2 ...        |
# t2 |         ...        |
# t3 |         ...        |
# t4 |         ...        |
# t5 |         ...        |
# t6 |         ...        |
class Image(Dataset):
    def __init__(self, y1, y2, img):
        self.X = torch.tensor(img, dtype=torch.float32)
        self.X = self.X.unsqueeze(1) # Convolutional networks expect a Channel value
        y1 = torch.tensor(y1, dtype=torch.float32)
        y2 = torch.tensor(y2, dtype=torch.float32)
        self.Y = torch.stack([y1, y2], dim=1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

# Feeding in the simulated data into the final format ...
formatted_y1 = Y.alpha.to_numpy()
formatted_y2 = Y.beta.to_numpy()
formatted_X = X.to_numpy().reshape(len(X), 6, 10000)
dataset = Image(formatted_y1, formatted_y2, formatted_X)

dataloader = DataLoader(
    dataset, 
    batch_size=16, # NN will look at 16 images at a time
    shuffle=True # mixes up the order of data for EVERY EPOCH
)

In [ ]:
import torch
import torch.nn as nn

class ConvNN(nn.Module):
    def __init__(self):
        super(ConvNN, self).__init__()
        
        # --- LAYER 1: Focus on the Time Dimension ---
        # Input shape:  (Batch, 1, 6, 10000)
        # Kernel (6,1) blends 6 timepoints together, but keeps genes independent
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=(6, 1), stride=1, padding='same')
        self.relu1 = nn.ReLU()
        
        # --- LAYER 2: Focus on Neighboring Genes ---
        # Input shape:  (Batch, 16, 6, 10000)
        # Kernel (1,5) scans across 5 genes at a time within the same timepoint
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(1, 5), stride=1, padding=(0, 2))
        self.relu2 = nn.ReLU()
        
        # --- LAYER 3: Downsampling the Gene Dimension ---
        # Instead of crushing the 6 timepoints, we only pool along the 10,000 genes
        # Reduces gene dimension from 10,000 to 5,000
        self.pool = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))
        
        # --- LAYER 4: Flatten & Fully Connected Layers ---
        # After Conv1, Conv2, and Pool, the shape becomes: (Batch, 32, 6, 5000)
        # Flattened size = 32 channels * 6 timepoints * 5000 genes = 960,000 features
        self.flatten = nn.Flatten()
        
        self.fc1 = nn.Linear(32 * 6 * 5000, 128)
        self.relu3 = nn.ReLU()
        
        # --- OUTPUT LAYER: Predict y1 and y2 ---
        # 128 features in -> 2 outputs out ([y1, y2])
        self.fc_out = nn.Linear(128, 2)

    def forward(self, x):
        # x shape: (Batch, 1, 6, 10000)
        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        x = self.pool(x)
        x = self.flatten(x)
        x = self.relu3(self.fc1(x))
        outputs = self.fc_out(x) 
        return outputs

In [ ]:
# 1. Initialize model, loss function, and optimizer
model = ConvNN()
criterion = nn.MSELoss()  # Good for regression tracking
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 2. Loop through training epochs
epochs = 10

for epoch in range(epochs):
    model.train()  # Put model in training mode
    running_loss = 0.0
    
    for batch_idx, (inputs, targets) in enumerate(dataloader):
        # Move data to the same device as the model
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Zero out the parameter gradients from the last step
        optimizer.zero_grad()
        
        # Forward pass: compute predicted outputs
        outputs = model(inputs)
        
        # Compute loss
        loss = criterion(outputs, targets)
        
        # Backward pass: compute gradient of the loss with respect to model parameters
        loss.backward()
        
        # Update weights
        optimizer.step()
        
        # Track statistics
        running_loss += loss.item() * inputs.size(0)
        
    epoch_loss = running_loss / len(dataloader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f}")

print("Training Complete!")

In [ ]:
model.eval()

In [ ]:
X_real = torch.tensor(dfW.to_numpy().reshape(1, 6, 10000), dtype=torch.float32).unsqueeze(1)
pred = model(x=X_real)


alpha = pred[0, 0].item()
beta = pred[0, 1].item()

alpha, beta

In [ ]:
alpha = 200
gamma_pdf = rng.gamma(shape=alpha, scale=1/(alpha+0.5), size=10_000)
sns.histplot(gamma_pdf)